In [1]:
import os
import shutil
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import pandas as pd
from tqdm import tqdm
from datetime import datetime

# ---------------- CONFIG ----------------
excel_path = r"C:\Users\chethan.m\Desktop\Sample\Sample Authority Revised.xlsx"
output_folder = r"C:\Users\chethan.m\Desktop\Sample\Authority"
temp_output_folder = r"C:\Users\chethan.m\Desktop\Sample\TEMP"
background_path = r"C:\Users\chethan.m\Desktop\Sample\Page Background (1).png"
signature_path = r"C:\Users\chethan.m\Desktop\Sample\Sign Of Axio (1).png"

MAX_WORKERS = max(1, cpu_count() - 1)
BATCH_SIZE = 3000
failure_log_path = os.path.join(output_folder, "failed_rows.csv")

os.makedirs(output_folder, exist_ok=True)
os.makedirs(temp_output_folder, exist_ok=True)

logging.basicConfig(filename=os.path.join(output_folder, 'pdf_generation.log'),
                    level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')

# ---------------- LETTER BODY ----------------
LETTER_BODY = """<b>PRIVATE & CONFIDENTIAL</b><br/><br/>
<b>AUTHORITY LETTER FOR COLLECTION VISIT</b><br/>
(Issued in compliance with RBI Fair Practices and Recovery Code of Conduct)<br/><br/>

Date: <b>«Date»</b><br/><br/>

To,<br/>
<b>«Customer_Name»</b><br/>
«Customer_Address»<br/>
«City_State_Pin_Code»<br/><br/>

Subject: <b>Authorization for Visit by Collection Representative</b><br/><br/>

Dear <b>«Customer_Name»</b>,<br/><br/>

This is to inform you that CapFloat Financial Services Private Limited (operating under the brand name “Axio”) has authorized the following representative to visit you regarding your loan account no. <b>«Loan_Account_Number»</b>, for the purpose of discussing and facilitating the collection of overdue payments, if any.<br/><br/>

<b>Authorized Representative Details:</b><br/>
• Name: <b>«Collectors_Full_Name»</b><br/>
• Employee / Agency ID: <b>«ID_Number»</b><br/>
• Collection Agency (if applicable): <b>«Agency_Name»</b><br/>
• Mobile Number: <b>«Collectors_Mobile_Number»</b><br/>
• Official ID Card No.: <b>«Issued_by_CapFloat__Agency»</b><br/>
• Authorized Visit Dates: <b>«Start_Date»</b> to <b>«End_Date»</b><br/>
• Permissible Visit Time: Between 08:00 AM and 07:00 PM<br/><br/>

The above representative is authorized to meet you solely for loan repayment discussion and assistance with regards to your loan.<br/><br/>

<b>Please note:</b><br/>
1. The representative is strictly instructed to conduct himself/herself in a professional, polite, and non-intrusive manner, as per RBI’s Guidelines on Fair Practices Code for Lenders and Circular on Digital Lending, 2022.<br/>
2. Cash payments beyond Rs.50,000 are not permitted & the same can be handed over only after due authentication of OTP sent on your mobile number as per our records.<br/>
3. All payments must be made only through official digital or banking channels (bank transfer, UPI, or cheque payable to CapFloat Financial Services Pvt. Ltd.).<br/>
4. The holder of this letter is not authorized to transact, negotiate or discuss any other matters apart from what is stated hereinabove.<br/>
5. In case of any misconduct, coercion, or inappropriate behaviour, you may immediately report the same to:<br/>
Email: <b>ask@axio.co.in</b><br/>
Helpline: <b>080 9879373 - 09:00 am to 09:00 pm</b><br/><br/>

We appreciate your cooperation and assure you of our commitment to transparency and fair conduct in all customer interactions.<br/><br/>

Regards,<br/>
"""

# ---------------- BACKGROUND FUNCTION ----------------
def add_background(canvas, doc):
    if background_path and os.path.exists(background_path):
        try:
            from reportlab.lib.pagesizes import A4
            page_width, page_height = A4
            canvas.saveState()
            canvas.drawImage(background_path, 0, 0, width=page_width, height=page_height,
                             preserveAspectRatio=True, mask='auto')
            canvas.restoreState()
        except Exception as e:
            logging.warning(f"Background image failed: {e}")

# ---------------- PDF GENERATION ----------------
def generate_pdf_from_row(row_dict, temp_output_folder):
    try:
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_JUSTIFY, TA_LEFT

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY, leading=12, fontSize=9))
        styles.add(ParagraphStyle(name='Left', alignment=TA_LEFT, leading=12, fontSize=9))

        def format_date(value):
            if pd.notna(value):
                try:
                    return pd.to_datetime(value).strftime("%d/%m/%Y")
                except Exception:
                    return str(value)
            return ""

        replacements = {
            "Date": format_date(row_dict.get("Date", "")),
            "Customer_Name": row_dict.get("Customer_Name", ""),
            "Customer_Address": row_dict.get("Customer_Address", ""),
            "City_State_Pin_Code": row_dict.get("City_State_Pin_Code", ""),
            "Loan_Account_Number": row_dict.get("Loan_Account_Number", ""),
            "Collectors_Full_Name": row_dict.get("Collectors_Full_Name", ""),
            "ID_Number": row_dict.get("ID_Number", ""),
            "Agency_Name": row_dict.get("Agency_Name", ""),
            "Collectors_Mobile_Number": row_dict.get("Collectors_Mobile_Number", ""),
            "Issued_by_CapFloat__Agency": row_dict.get("Issued_by_CapFloat__Agency", ""),
            "Start_Date": format_date(row_dict.get("Start_Date", "")),
            "End_Date": format_date(row_dict.get("End_Date", "")),
        }

        letter_text = LETTER_BODY
        for key, val in replacements.items():
            letter_text = letter_text.replace(f"«{key}»", str(val))

        filename = f"{row_dict.get('Loan_Account_Number', 'unknown')}.pdf"
        tmp_path = os.path.join(temp_output_folder, filename)

        doc = SimpleDocTemplate(tmp_path, pagesize=A4,
                                rightMargin=50, leftMargin=50,
                                topMargin=70, bottomMargin=60)

        story = []
        # Main letter body
        for para in letter_text.split("<br/><br/>"):
            if para.strip():
                story.append(Paragraph(para.strip(), styles['Justify']))
                story.append(Spacer(1, 10))

        # Signature section
        story.append(Spacer(1, 12))
        if signature_path and os.path.exists(signature_path):
            story.append(Image(signature_path, width=100, height=30, hAlign='LEFT'))
            story.append(Spacer(1, 6))
        story.append(Paragraph("<b>Authorised Signatory</b>", styles['Left']))
        story.append(Paragraph("CapFloat Financial Services Pvt. Ltd.", styles['Left']))

        doc.build(story, onFirstPage=add_background)
        return True, filename, tmp_path

    except Exception as e:
        return False, row_dict.get("Loan_Account_Number", "unknown"), str(e)

# ---------------- BATCH RUNNER ----------------
def run_in_batches(df, temp_output_folder, output_folder,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE):
    rows = df.to_dict(orient="records")
    failures = []

    for batch_start in range(0, len(rows), batch_size):
        batch = rows[batch_start: batch_start + batch_size]
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(generate_pdf_from_row, row, temp_output_folder): row for row in batch}

            for fut in tqdm(as_completed(futures), total=len(futures), desc="Generating PDFs"):
                row = futures[fut]
                try:
                    success, filename, result = fut.result()
                    if success:
                        shutil.move(result, os.path.join(output_folder, filename))
                    else:
                        logging.error(f"Failed {filename}: {result}")
                        failures.append({**row, "error": result})
                except Exception as e:
                    failures.append({**row, "error": str(e)})

    if failures:
        pd.DataFrame(failures).to_csv(failure_log_path, index=False)
        print(f"\n⚠️ Some PDFs failed. Check '{failure_log_path}' for details.")
    else:
        print("\n✅ All PDFs generated successfully!")

# ---------------- MAIN ----------------
if __name__ == "__main__":
    df = pd.read_excel(excel_path)
    df.columns = [col.strip().replace(" ", "_").replace("-", "_") for col in df.columns]
    run_in_batches(df, temp_output_folder, output_folder)


Generating PDFs: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.41s/it]


✅ All PDFs generated successfully!
